In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import datasets
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf

In [3]:
from datasets import load_dataset
import pandas as pd
ds=load_dataset("Tobi-Bueck/customer-support-tickets")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

aa_dataset-tickets-multi-lang-5-2-50-ver(…):   0%|          | 0.00/26.0M [00:00<?, ?B/s]

(…)set-tickets-german_normalized_50_5_2.csv: 0.00B [00:00, ?B/s]

dataset-tickets-multi-lang-4-20k.csv:   0%|          | 0.00/18.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/61765 [00:00<?, ? examples/s]

In [4]:
df=pd.DataFrame(ds['train'])
df.to_csv("data.csv")

In [5]:
df=df[['body','queue','language']]
df.head()

,body,queue,language
0,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Technical Support,de
1,"Dear Customer Support Team,\n\nI am writing to...",Technical Support,en
2,"Dear Customer Support Team,\n\nI hope this mes...",Returns and Exchanges,en
3,"Dear Customer Support Team,\n\nI hope this mes...",Billing and Payments,en
4,"Dear Support Team,\n\nI hope this message reac...",Sales and Pre-Sales,en


In [11]:
df['body'].isnull().sum()
df['body']=df['body'].fillna("")
df['body']=df['body'].fillna("").str.lower()

In [7]:
import re

df['body']=df['body'].apply(lambda x: re.sub(r'nn+',' ',x))
df['body']=df['body'].apply(lambda x: re.sub(r'\s+', ' ',x).strip())
df['body']=df['body'].apply(lambda x: re.sub(r'([a-z])([A-Z])',r'\1 \2',x))

In [8]:
import re
import string

df['body']=df['body'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

In [9]:
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [12]:
from nltk.tokenize import word_tokenize
df['tokens']=df['body'].apply(word_tokenize)
df.head()

,body,queue,language,tokens
0,sehr geehrtes supportteamnnich möchte einen gr...,Technical Support,de,"[sehr, geehrtes, supportteamnnich, möchte, ein..."
1,dear customer support teamnni am writing to re...,Technical Support,en,"[dear, customer, support, teamnni, am, writing..."
2,dear customer support teamnni hope this messag...,Returns and Exchanges,en,"[dear, customer, support, teamnni, hope, this,..."
3,dear customer support teamnni hope this messag...,Billing and Payments,en,"[dear, customer, support, teamnni, hope, this,..."
4,dear support teamnni hope this message reaches...,Sales and Pre-Sales,en,"[dear, support, teamnni, hope, this, message, ..."


In [13]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

english=set(stopwords.words('english'))
german=set(stopwords.words('german'))

df['tokens']=df['tokens'].apply(
    lambda words:[w.lower() for w in words if w.isalpha()]
)
df.loc[df['language'] == 'en', 'tokens'] = df.loc[df['language'] == 'en', 'tokens'].apply(
    lambda words: [w for w in words if w not in english]
)

df.loc[df['language'] == 'de', 'tokens'] = df.loc[df['language'] == 'de', 'tokens'].apply(
    lambda words: [w for w in words if w not in german]
)

df.head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,body,queue,language,tokens
0,sehr geehrtes supportteamnnich möchte einen gr...,Technical Support,de,"[geehrtes, supportteamnnich, möchte, gravieren..."
1,dear customer support teamnni am writing to re...,Technical Support,en,"[dear, customer, support, teamnni, writing, re..."
2,dear customer support teamnni hope this messag...,Returns and Exchanges,en,"[dear, customer, support, teamnni, hope, messa..."
3,dear customer support teamnni hope this messag...,Billing and Payments,en,"[dear, customer, support, teamnni, hope, messa..."
4,dear support teamnni hope this message reaches...,Sales and Pre-Sales,en,"[dear, support, teamnni, hope, message, reache..."


In [14]:
df['tokens']=df['tokens'].apply(lambda words:[w.lower() for w in words])
df.head()

,body,queue,language,tokens
0,sehr geehrtes supportteamnnich möchte einen gr...,Technical Support,de,"[geehrtes, supportteamnnich, möchte, gravieren..."
1,dear customer support teamnni am writing to re...,Technical Support,en,"[dear, customer, support, teamnni, writing, re..."
2,dear customer support teamnni hope this messag...,Returns and Exchanges,en,"[dear, customer, support, teamnni, hope, messa..."
3,dear customer support teamnni hope this messag...,Billing and Payments,en,"[dear, customer, support, teamnni, hope, messa..."
4,dear support teamnni hope this message reaches...,Sales and Pre-Sales,en,"[dear, support, teamnni, hope, message, reache..."


In [15]:
df['tokens']=df['tokens'].apply(lambda words: [w for w in words if w.isalpha()])
df['tokens'].head()

,tokens
0,"[geehrtes, supportteamnnich, möchte, gravieren..."
1,"[dear, customer, support, teamnni, writing, re..."
2,"[dear, customer, support, teamnni, hope, messa..."
3,"[dear, customer, support, teamnni, hope, messa..."
4,"[dear, support, teamnni, hope, message, reache..."


In [16]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [17]:
from nltk.stem import WordNetLemmatizer

lemma=WordNetLemmatizer()
df['tokens']=df['tokens'].apply(
    lambda words:[lemma.lemmatize(w) for w in words]
)
df.head()

,body,queue,language,tokens
0,sehr geehrtes supportteamnnich möchte einen gr...,Technical Support,de,"[geehrtes, supportteamnnich, möchte, gravieren..."
1,dear customer support teamnni am writing to re...,Technical Support,en,"[dear, customer, support, teamnni, writing, re..."
2,dear customer support teamnni hope this messag...,Returns and Exchanges,en,"[dear, customer, support, teamnni, hope, messa..."
3,dear customer support teamnni hope this messag...,Billing and Payments,en,"[dear, customer, support, teamnni, hope, messa..."
4,dear support teamnni hope this message reaches...,Sales and Pre-Sales,en,"[dear, support, teamnni, hope, message, reach,..."


In [18]:
df['tokens']=df['tokens'].apply(lambda words: " ".join(words))

In [ ]:
print(df.columns)

Index(['body', 'queue', 'language', 'tokens'], dtype='object')


In [19]:
x=df['tokens']
y=df['queue']

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf=TfidfVectorizer(max_features=5000)
x_train_vec=tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

In [ ]:
df.to_csv("clean_data.csv")

In [ ]:
# df = ds['train'].to_pandas()

In [21]:
from sklearn.linear_model import LogisticRegression
m1=LogisticRegression(max_iter=1000,class_weight='balanced')
m1.fit(x_train_vec,y_train)
y_pred=m1.predict(X_test_vec)

In [22]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.42920747996438113
Classification Report:
                                         precision    recall  f1-score   support

           Arts & Entertainment/Movies       0.90      0.80      0.84        54
            Arts & Entertainment/Music       0.67      0.74      0.70        77
          Autos & Vehicles/Maintenance       0.53      0.63      0.58        62
                Autos & Vehicles/Sales       0.61      0.65      0.63        66
            Beauty & Fitness/Cosmetics       0.67      0.80      0.73        55
     Beauty & Fitness/Fitness Training       0.71      0.65      0.68        60
                  Billing and Payments       0.73      0.67      0.70       990
            Books & Literature/Fiction       0.67      0.69      0.68        52
        Books & Literature/Non-Fiction       0.79      0.76      0.77        63
   Business & Industrial/Manufacturing       0.82      0.73      0.77        73
                      Customer Service       0.36      0.22      

In [23]:
from sklearn.naive_bayes import MultinomialNB

m2=MultinomialNB()
m2.fit(x_train_vec,y_train)
y_pred=m2.predict(X_test_vec)

In [24]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.43446935966971584
Classification Report:
                                         precision    recall  f1-score   support

           Arts & Entertainment/Movies       0.95      0.37      0.53        54
            Arts & Entertainment/Music       1.00      0.26      0.41        77
          Autos & Vehicles/Maintenance       0.77      0.27      0.40        62
                Autos & Vehicles/Sales       0.38      0.67      0.48        66
            Beauty & Fitness/Cosmetics       0.73      0.55      0.62        55
     Beauty & Fitness/Fitness Training       0.93      0.42      0.57        60
                  Billing and Payments       0.88      0.56      0.68       990
            Books & Literature/Fiction       0.72      0.50      0.59        52
        Books & Literature/Non-Fiction       0.52      0.67      0.58        63
   Business & Industrial/Manufacturing       0.63      0.71      0.67        73
                      Customer Service       0.28      0.45      

In [25]:
from sklearn.svm import LinearSVC

m3=LinearSVC()
m3.fit(x_train_vec,y_train)
y_pred=m3.predict(X_test_vec)

In [26]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.5452116894681454
Classification Report:
                                         precision    recall  f1-score   support

           Arts & Entertainment/Movies       0.87      0.76      0.81        54
            Arts & Entertainment/Music       0.73      0.74      0.74        77
          Autos & Vehicles/Maintenance       0.60      0.58      0.59        62
                Autos & Vehicles/Sales       0.60      0.65      0.62        66
            Beauty & Fitness/Cosmetics       0.76      0.80      0.78        55
     Beauty & Fitness/Fitness Training       0.79      0.63      0.70        60
                  Billing and Payments       0.77      0.71      0.74       990
            Books & Literature/Fiction       0.73      0.69      0.71        52
        Books & Literature/Non-Fiction       0.74      0.83      0.78        63
   Business & Industrial/Manufacturing       0.86      0.78      0.82        73
                      Customer Service       0.40      0.39      0

In [27]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

In [28]:
from sklearn.ensemble import RandomForestClassifier

m4=RandomForestClassifier(class_weight=class_weights)
m4.fit(x_train_vec,y_train)
y_pred=m4.predict(X_test_vec)

In [29]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.7094632882700559
Classification Report:
                                         precision    recall  f1-score   support

           Arts & Entertainment/Movies       0.93      0.80      0.86        54
            Arts & Entertainment/Music       0.79      0.70      0.74        77
          Autos & Vehicles/Maintenance       0.66      0.66      0.66        62
                Autos & Vehicles/Sales       0.59      0.71      0.65        66
            Beauty & Fitness/Cosmetics       0.73      0.89      0.80        55
     Beauty & Fitness/Fitness Training       0.85      0.68      0.76        60
                  Billing and Payments       0.93      0.78      0.85       990
            Books & Literature/Fiction       0.72      0.75      0.74        52
        Books & Literature/Non-Fiction       0.78      0.75      0.76        63
   Business & Industrial/Manufacturing       0.80      0.73      0.76        73
                      Customer Service       0.66      0.66      0

In [30]:
import pickle

with open("model1.pkl","wb") as f:
  pickle.dump((m4,tfidf),f)

In [31]:
text=df['tokens']

In [32]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
tokenizer = Tokenizer(num_words=20000,oov_token="<OOV>")
tokenizer.fit_on_texts(text)
sequences = tokenizer.texts_to_sequences(text)
pad=pad_sequences(sequences,maxlen=150,padding='post',truncating='post')

from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
y=le.fit_transform(df['queue'])

In [33]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    pad,
    y,
    test_size=0.2,
    random_state=42
)

In [34]:
df['tokens'] = df['tokens'].apply(lambda x: x.split())

In [ ]:
print(df['tokens'].head())

0    [geehrtes, supportteamnnich, möchte, gravieren...
1    [dear, customer, support, teamnni, writing, re...
2    [dear, customer, support, teamnni, hope, messa...
3    [dear, customer, support, teamnni, hope, messa...
4    [dear, support, teamnni, hope, message, reach,...
Name: tokens, dtype: object


In [ ]:
unique_words = set()

for tokens in df['tokens']:
    unique_words.update(tokens)

print("Total Unique Words:", len(unique_words))

Total Unique Words: 48836


In [35]:
from collections import Counter

counter = Counter()

for tokens in df['tokens']:
    counter.update(tokens)

# Vocabulary dictionary
vocab = {
    '<PAD>': 0,
    '<UNK>': 1
}

# Add words to vocab
for word, freq in counter.items():

    # Keep words appearing at least 2 times
    if freq >= 2:
        vocab[word] = len(vocab)

vocab_size = len(vocab)

print("Vocabulary Size:", vocab_size)

Vocabulary Size: 28204


In [ ]:
!pip install datasets nltk scikit-learn torch

  Using cached torch-2.11.0-cp310-cp310-win_amd64.whl (114.5 MB)
  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)
     ---------------------------------------- 0.0/536.2 kB ? eta -:--:--
      --------------------------------------- 10.2/536.2 kB ? eta -:--:--
     -- ---------------------------------- 41.0/536.2 kB 393.8 kB/s eta 0:00:02
     ---- -------------------------------- 71.7/536.2 kB 438.9 kB/s eta 0:00:02
     ------ ----------------------------- 102.4/536.2 kB 535.8 kB/s eta 0:00:01
     --------- -------------------------- 143.4/536.2 kB 568.9 kB/s eta 0:00:01
     --------- -------------------------- 143.4/536.2 kB 568.9 kB/s eta 0:00:01
     ---------- ------------------------- 153.6/536.2 kB 482.7 kB/s eta 0:00:01
     ----------- ------------------------ 174.1/536.2 kB 476.3 kB/s eta 0:00:01
     ------------ ----------------------- 184.3/536.2 kB 446.4 kB/s eta 0:00:01
     ------------- -------------------


[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from collections import Counter

In [37]:
class LSTMClassifier(nn.Module):

    def __init__(self,
                 vocab_size,
                 embedding_dim,
                 hidden_dim,
                 output_dim,
                 num_layers=2,
                 dropout=0.3):

        super(LSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_dim * 2, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        embedded = self.embedding(x)

        lstm_out, (hidden, cell) = self.lstm(embedded)

        # Concatenate forward + backward hidden states
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)

        hidden = self.dropout(hidden)

        output = self.fc(hidden)

        return output

In [38]:
from torch.utils.data import Dataset, DataLoader
import torch

class TicketDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return self.X[idx], self.y[idx]

In [39]:
train_dataset = TicketDataset(X_train, y_train)

test_dataset = TicketDataset(X_test, y_test)

In [40]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32
)

In [41]:
num_classes = len(le.classes_)

print(num_classes)

52


In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes=52

model = LSTMClassifier(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_dim=128,
    output_dim=num_classes
)

model = model.to(device)

print(model)

LSTMClassifier(
  (embedding): Embedding(28204, 128, padding_idx=0)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (fc): Linear(in_features=256, out_features=52, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


In [43]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to tensor
class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
).to(device)

# Loss function with class weights
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 18

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for texts, labels in train_loader:

        texts = texts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(texts)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

Epoch [1/18], Loss: 3.4451
Epoch [2/18], Loss: 2.8804
Epoch [3/18], Loss: 2.0772
Epoch [4/18], Loss: 1.3714
Epoch [5/18], Loss: 0.9239
Epoch [6/18], Loss: 0.6888
Epoch [7/18], Loss: 0.5369
Epoch [8/18], Loss: 0.4556
Epoch [9/18], Loss: 0.4289
Epoch [10/18], Loss: 0.3498
Epoch [11/18], Loss: 0.3154
Epoch [12/18], Loss: 0.3050
Epoch [13/18], Loss: 0.2917
Epoch [14/18], Loss: 0.2540
Epoch [15/18], Loss: 0.2421
Epoch [16/18], Loss: 0.2234
Epoch [17/18], Loss: 0.2101
Epoch [18/18], Loss: 0.2014


In [48]:
num_epochs = 20

for epoch in range(num_epochs):

    # TRAINING
    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for texts, labels in train_loader:

        texts = texts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(texts)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)

        train_correct += (predicted == labels).sum().item()

    train_accuracy = train_correct / train_total

    # VALIDATION
    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for texts, labels in test_loader:

            texts = texts.to(device)
            labels = labels.to(device)

            outputs = model(texts)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_accuracy = val_correct / val_total

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {train_loss/len(train_loader):.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

Epoch 1 | Train Loss: 0.1688 | Train Acc: 0.6486 | Val Acc: 0.4872
Epoch 2 | Train Loss: 0.1742 | Train Acc: 0.6530 | Val Acc: 0.4856
Epoch 3 | Train Loss: 0.1677 | Train Acc: 0.6650 | Val Acc: 0.4936
Epoch 4 | Train Loss: 0.1474 | Train Acc: 0.6822 | Val Acc: 0.4997
Epoch 5 | Train Loss: 0.1540 | Train Acc: 0.6853 | Val Acc: 0.5047
Epoch 6 | Train Loss: 0.1480 | Train Acc: 0.6927 | Val Acc: 0.5146
Epoch 7 | Train Loss: 0.1387 | Train Acc: 0.7088 | Val Acc: 0.5212
Epoch 8 | Train Loss: 0.1347 | Train Acc: 0.7221 | Val Acc: 0.5123
Epoch 9 | Train Loss: 0.1268 | Train Acc: 0.7317 | Val Acc: 0.5412
Epoch 10 | Train Loss: 0.1281 | Train Acc: 0.7386 | Val Acc: 0.5328
Epoch 11 | Train Loss: 0.1177 | Train Acc: 0.7462 | Val Acc: 0.5348
Epoch 12 | Train Loss: 0.1182 | Train Acc: 0.7501 | Val Acc: 0.5440
Epoch 13 | Train Loss: 0.1170 | Train Acc: 0.7650 | Val Acc: 0.5400
Epoch 14 | Train Loss: 0.1095 | Train Acc: 0.7745 | Val Acc: 0.5484
Epoch 15 | Train Loss: 0.1048 | Train Acc: 0.7816 | Val A

In [49]:
model.eval()

predictions = []
actuals = []

with torch.no_grad():

    for texts, labels in test_loader:

        texts = texts.to(device)

        outputs = model(texts)

        _, preds = torch.max(outputs, 1)

        predictions.extend(preds.cpu().numpy())

        actuals.extend(labels.numpy())

In [50]:
accuracy = accuracy_score(actuals, predictions)

print("Accuracy:", accuracy)

print(classification_report(actuals, predictions))

Accuracy: 0.5670687282441512
              precision    recall  f1-score   support

           0       0.80      0.83      0.82        54
           1       0.80      0.82      0.81        77
           2       0.76      0.61      0.68        62
           3       0.73      0.65      0.69        66
           4       0.91      0.87      0.89        55
           5       0.88      0.73      0.80        60
           6       0.82      0.76      0.79       990
           7       0.64      0.90      0.75        52
           8       0.79      0.83      0.81        63
           9       0.93      0.78      0.85        73
          10       0.41      0.50      0.45      1460
          11       0.92      0.86      0.89        66
          12       0.82      0.74      0.78        54
          13       0.83      0.70      0.76        71
          14       0.77      0.78      0.78        60
          15       0.69      0.67      0.68        67
          16       0.41      0.45      0.43       12